In [1]:
#For my use - To check what NoSQL DBs are there
import pymongo

client = pymongo.MongoClient("127.0.0.1", 27017)
databases = client.database_names()
print("The databases in the MongoDB server are: ")
print(databases)
client.close()

The databases in the MongoDB server are: 
['CodeNova', 'DHS', 'admin', 'entertainment', 'local', 'school']


c:\program files\python37\lib\site-packages\ipykernel_launcher.py:5: DeprecationWarning: database_names is deprecated. Use list_database_names instead.
  """


In [1]:
#Task 1.1 - create NoSQL DB & collection and load the JSON data into the collection
import pymongo, json

client = pymongo.MongoClient("127.0.0.1", 27017) 
db = client.get_database("DHS") # 1 mark - create database successfully
db.drop_collection("IT_Devices") # 1 mark - deleting IT_Devices collection and creating new IT_Devices collection
coll = db.get_collection("IT_Devices") #create collection/table

with open('IT_Devices.json', 'r') as file: # read from JSON file (no marks as sample code provided)
    data = json.load(file)

coll.insert_many(data) # 1 mark - insert data into collection/table

result = coll.find() # 1 mark - retrieve all documents in collection/table to display 
print("All documents in IT Resources collection:")
for document in result:
    print(document)
    
client.close()

All documents in IT Resources collection:
{'_id': ObjectId('68aabc88ad45635c4ad18bc4'), 'Serial_Number': 'LAP-001', 'Type': 'Laptop', 'Brand': 'HP', 'Cost': 1099, 'Date_Of_Purchase': '2022-02-10', 'Weight': 1.35, 'Screen_Size': 14}
{'_id': ObjectId('68aabc88ad45635c4ad18bc5'), 'Serial_Number': 'LAP-002', 'Type': 'Laptop', 'Brand': 'Lenovo', 'Cost': 1599, 'Date_Of_Purchase': '2023-09-18', 'Weight': 2.45, 'Screen_Size': 16}
{'_id': ObjectId('68aabc88ad45635c4ad18bc6'), 'Serial_Number': 'LAP-003', 'Type': 'Laptop', 'Brand': 'Apple', 'Cost': 1899, 'Date_Of_Purchase': '2024-05-25', 'Weight': 1.25, 'Screen_Size': 14}
{'_id': ObjectId('68aabc88ad45635c4ad18bc7'), 'Serial_Number': 'LAP-004', 'Type': 'Laptop', 'Brand': 'Dell', 'Cost': 1249, 'Date_Of_Purchase': '2022-01-08', 'Weight': 1.55, 'Screen_Size': 14}
{'_id': ObjectId('68aabc88ad45635c4ad18bc8'), 'Serial_Number': 'LAP-005', 'Type': 'Laptop', 'Brand': 'ASUS', 'Cost': 2199, 'Date_Of_Purchase': '2023-12-03', 'Weight': 2.6, 'Screen_Size': 17

In [2]:
#Task 1.2 - update all the HP products name to the full name: Hewlett-Packard
import pymongo
client = pymongo.MongoClient("127.0.0.1", 27017)
db = client.get_database("DHS")
coll = db.get_collection("IT_Devices") # 1 mark for connecting to the database and collection

#updating HP to Hewlett-Packard
search = {'Brand':"HP"}
update = {'$set':{'Brand':"Hewlett-Packard"}}  
coll.update_many(search, update) #2 marks - updating all HP values into Hewlett-Packard

#display all the laptops that weigh less than 2kg and all the tablets that have at least 8000W Battery Capacity (simple formatting of display required)
query2 = {'$or':[{'$and':[{'Type': 'Laptop'}, {'Weight': {'$lt': 2}}]},{'$and':[{'Type': 'Tablet'}, {'Battery_Capacity': {'$gte': 8000}}]}]} #2 marks - correct query.
result = coll.find(query2) #1 mark - retrieve the correct documents successfully
for document in result:
    #1 mark - display all relavant devices in the correct format
    if document['Type'] == "Laptop":
        print(f"Serial No: {document['Serial_Number']}, Type: {document['Type']}, Brand: {document['Brand']}, Weight: {document['Weight']}")
    else:
        print(f"Serial No: {document['Serial_Number']}, Type: {document['Type']}, Brand: {document['Brand']}, Battery Capacity: {document['Battery_Capacity']}")
    
client.close()

Serial No: LAP-001, Type: Laptop, Brand: Hewlett-Packard, Weight: 1.35
Serial No: LAP-003, Type: Laptop, Brand: Apple, Weight: 1.25
Serial No: LAP-004, Type: Laptop, Brand: Dell, Weight: 1.55
Serial No: LAP-006, Type: Laptop, Brand: Acer, Weight: 1.45
Serial No: TAB-001, Type: Tablet, Brand: Samsung, Battery Capacity: 8000
Serial No: TAB-002, Type: Tablet, Brand: Apple, Battery Capacity: 9720


In [3]:
#Task 1.3 - create the Relational DB and transfer the data from NoSQL DB into the respective tables in Relational DB
import sqlite3, pymongo

# connecting to NoSQL DB
client = pymongo.MongoClient("127.0.0.1", 27017)
db = client.get_database("DHS")
coll = db.get_collection("IT_Devices")

connection = sqlite3.connect("DHS.db") # 1 mark - creating DB successfully

#create tables - recommended to put the IF NOT EXISTS part else will cause error when doing repeated testing of this code block
#1 mark for creating Device Table correctly [correct field types & PK], 3 marks for creating Laptop and Tablet tables correctly [correct field types & PK/FK] and 1 mark for correct table creation sequence [Device table must be created first]
connection.execute('''CREATE TABLE IF NOT EXISTS 'Device' (
                    'Serial_Number' TEXT,
                    'Type' TEXT NOT NULL,
                    'Brand' TEXT NOT NULL,
                    'Cost' INTEGER NOT NULL,
                    'Date_Of_Purchase' TEXT NOT NULL,
                    PRIMARY KEY('Serial_Number'));''')

connection.execute('''CREATE TABLE IF NOT EXISTS 'Laptop' (
                    'Serial_Number' TEXT,
                    'Weight' REAL NOT NULL,
                    'Screen_Size' INTEGER NOT NULL,
                    FOREIGN KEY('Serial_Number') REFERENCES 'Device'('Serial_Number'),
                    PRIMARY KEY('Serial_Number'));''')

connection.execute('''CREATE TABLE IF NOT EXISTS 'Tablet' (
                    'Serial_Number' TEXT,
                    'Battery_Capacity' INTEGER NOT NULL,
                    FOREIGN KEY('Serial_Number') REFERENCES 'Device'('Serial_Number'),
                    PRIMARY KEY('Serial_Number'));''')

#retrieve all data from NoSQL DB to insert into Relational DB
result = coll.find() # retrieve all documents in NoSQL collection/table 
for document in result:
    #inserting into the respective tables in Relational DB
    #2 marks - inserting data correctly into Device table [correct sql query & correct values given]
    connection.execute("INSERT INTO Device (Serial_Number, Type, Brand, Cost, Date_Of_Purchase) VALUES (?,?,?,?,?)", (document['Serial_Number'],document['Type'],document['Brand'],document['Cost'],document['Date_Of_Purchase']))
    
    #2 marks insert data correctly into Laptop and Tablet table respectively [correct condition to decide which table to store the data & correct sql query/values]
    if document['Type'] == "Laptop":
        connection.execute("INSERT INTO Laptop (Serial_Number, Weight, Screen_Size) VALUES (?,?,?)", (document['Serial_Number'], document['Weight'], document['Screen_Size']))
    else:
        connection.execute("INSERT INTO Tablet (Serial_Number, Battery_Capacity) VALUES (?,?)", (document['Serial_Number'], document['Battery_Capacity']))

connection.commit() # 1 mark - successfully inserted any data  from NoSQL database into the relational database
connection.close()

In [1]:
#Task 1.4 - find all the devices older than 1000 days old and log into a csv file

import sqlite3, datetime, csv

connection = sqlite3.connect("DHS.db") # create database connection

devices_to_be_updated = []

cursor = connection.execute('''SELECT * FROM Device;''') # 1 mark - correct SQL query to extract data from relational db

records = cursor.fetchall()
for record in records:
    #print(record)
    #print(record[4])
    Date_Of_Purchase_text = record[4]
    year, month, day = Date_Of_Purchase_text.split("-")
    #print(year, month, day)
    Date_Of_Purchase_datetime = datetime.datetime(int(year), int(month), int(day))
    length = datetime.datetime.now() - Date_Of_Purchase_datetime #2 marks - correct calculation of the age of the device (in days)
    #print(length.days) #to get the number of days, ignore the time differences
    if length.days > 1000: #1 mark - correct identification of old devices
        devices_to_be_updated.append([record[0], record[1], record[4], length.days])
      

#1 mark - successfully writing to csv file
#1 mark - all old devices (laptop and tablet) are recorded correctly
with open("Old_devices.csv", "a", newline = "") as outfile:
    writer = csv.writer(outfile, delimiter=",")
    writer.writerows(devices_to_be_updated)
    
connection.close()